# Bài 19: Cách mọi thứ kết nối

Bạn đã sử dụng `python output/chat.py` để tạo bài viết. Nhưng điều gì xảy ra **đằng sau** khi bạn nói "Tạo bài viết về SEO"?

Trong bài học này, chúng ta sẽ theo dõi **toàn bộ hành trình của một yêu cầu** — từ lúc bạn gõ tin nhắn đến lúc bài viết xuất hiện trong database. Không cần chạy code, không tốn API — chỉ là tổng quan cách các phần kết nối với nhau.

Giống như xem **bản thiết kế** của tòa nhà mà bạn đã đi qua.

## Hành Trình Của Một Yêu Cầu

Khi bạn gõ "Tạo bài viết về on-page SEO" trong chat, đây là những gì xảy ra:

```
Bạn (tin nhắn chat)
  │
  ▼
chat.py ─── Team Leader (Opus) quyết định giao cho ai
  │
  ▼
tools/workspace.py ─── Content Creator gọi create_article()
  │
  ▼
pipeline.py ─── Chạy 4 bước agent: nghiên cứu → dàn ý → viết → làm giàu
  │
  ▼
agents/ ─── Mỗi file agent thực hiện công việc chuyên biệt
  │
  ▼
tools/airtable.py ─── Trạng thái và nội dung được lưu ở mỗi bước
  │
  ▼
content/ ─── Bài viết cuối cùng được lưu dưới dạng file .md
```

Mỗi file có **một trách nhiệm duy nhất**. Hãy đi qua từng file một.

## chat.py — Cửa vào

`chat.py` là **điểm vào** — file bạn chạy với `python output/chat.py`.

Nó làm ba việc:
1. Kiểm tra API key (`ANTHROPIC_API_KEY`, `XAI_API_KEY`)
2. Kiểm tra cấu hình Airtable
3. Import team từ `agents/team.py` và khởi chạy chat

Team được khai báo trong `agents/team.py`, mỗi thành viên nằm trong file riêng:
- **Content Creator** (`agents/content_creator.py`) — tạo bài viết, chạy lại bài lỗi, xử lý hàng loạt
- **Status Tracker** (`agents/status_tracker.py`) — kiểm tra trạng thái bài viết, chi tiết, lịch sử
- **AIO Analyst** (`agents/aio_analyst.py`) — phân tích AI Overview, tối ưu nội dung cho AIO
- **Team Leader** (`agents/team.py`) — Claude Opus, đọc tin nhắn và ủy thác

Mỗi thành viên có **tools riêng** từ `tools/workspace.py`. Leader không gọi tools trực tiếp — chỉ ủy thác cho đúng thành viên.

Đây là mô hình **quản lý dự án**: bạn nói chuyện với một người, họ biết phân việc cho ai.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

# Hãy xem cấu trúc team trong chat.py
chat_path = os.path.abspath("../../output/chat.py")
with open(chat_path, "r", encoding="utf-8") as f:
    chat_code = f.read()

print(f"chat.py ({len(chat_code.splitlines())} dòng)\n")

# Hiển thị định nghĩa các thành viên team
for i, line in enumerate(chat_code.splitlines(), 1):
    print(f"  {i:3d} | {line}")

## tools/workspace.py — Cầu Nối

Khi Content Creator nhận lệnh "Tạo bài viết về on-page SEO", nó gọi tool `create_article()`.

`tools/workspace.py` là **cầu nối** giữa team chat và pipeline. Nó chứa các hàm Python thuần túy:
1. Nhận tham số đơn giản (chủ đề, từ khóa)
2. Gọi hàm pipeline và database
3. Trả về chuỗi JSON để AI có thể hiểu

Tại sao cần cầu nối? Các thành viên AI cần **hàm đơn giản, được mô tả rõ ràng** làm tools. Pipeline có logic phức tạp hơn. Cầu nối giữ cả hai phía gọn gàng.

In [ ]:
# Hãy xem hàm create_article trong tools/workspace.py
ws_path = os.path.abspath("../../output/tools/workspace.py")
with open(ws_path, "r", encoding="utf-8") as f:
    ws_code = f.read()

# Hiển thị hàm create_article (khoảng 48 dòng đầu)
lines = ws_code.splitlines()
print("tools/workspace.py — hàm create_article():\n")
for i, line in enumerate(lines[:48], 1):
    print(f"  {i:3d} | {line}")

## pipeline.py — Bộ máy

`pipeline.py` là **trái tim** của sản phẩm. Hàm `run_content_pipeline()` chạy 4 bước tuần tự:

```
1. Nghiên cứu  → research_agent tìm kiếm web (Claude Sonnet + DuckDuckGo)
2. Dàn ý       → outline_agent tạo cấu trúc (Claude Sonnet + output_schema)
3. Viết bài    → writer_agent viết bài (Grok-4, Markdown thuần)
4. Làm giàu    → image_agent tìm hình ảnh (Claude Sonnet, tuỳ chọn)
```

Giữa mỗi bước, pipeline:
- Cập nhật trạng thái bài viết trong Airtable
- Bắt lỗi để một bài viết thất bại không crash toàn bộ

Luồng trạng thái:
```
queued → researching → outlining → writing → enriching → review
```

Nếu bất kỳ bước nào lỗi: `status → error` kèm `error_message` được lưu lại.

In [ ]:
# Hãy xem code pipeline
pipeline_path = os.path.abspath("../../output/pipeline.py")
with open(pipeline_path, "r", encoding="utf-8") as f:
    pipeline_code = f.read()

print(f"pipeline.py ({len(pipeline_code.splitlines())} dòng)\n")

# Hiển thị hàm run_content_pipeline (logic chính)
lines = pipeline_code.splitlines()
for i, line in enumerate(lines[:161], 1):
    print(f"  {i:3d} | {line}")

## tools/airtable.py — Nơi lưu dữ liệu

`tools/airtable.py` quản lý database Airtable. Nó cung cấp các hàm đơn giản:

| Hàm | Mục đích |
|-----|----------|
| `create_article(topic, keywords)` | Tạo bài viết mới (status = 'queued') |
| `update_article_status(id, status, **fields)` | Cập nhật trạng thái + bất kỳ trường nào |
| `get_article(id)` | Lấy một bài viết dạng dict |
| `list_articles(status, batch_id)` | Liệt kê bài viết với bộ lọc tùy chọn |
| `save_aio_analysis(article_id, keyword, ...)` | Ghi lại kết quả phân tích AI Overview |
| `get_aio_analyses(article_id)` | Xem lịch sử phân tích AIO |

Mọi hàm trả về **dict Python thuần** — không ORM, không phức tạp. Nếu Airtable chưa cấu hình, các hàm fallback an toàn (trả về dict/list rỗng).

In [ ]:
# Hãy xem một article dict trông như thế nào
from tools.airtable import get_article, list_articles

# Hiển thị cấu trúc của một article dict
articles = list_articles()
if articles:
    article = articles[0]
    print("Một article dict trông như thế này:\n")
    for key, value in article.items():
        val_preview = str(value)[:60] + "..." if value and len(str(value)) > 60 else value
        print(f"  {key:<20} = {val_preview}")
else:
    print("Chưa có bài viết nào trong database.")
    print("Một article dict có các key:")
    print("  id, topic, target_keywords, status, outline_json,")
    print("  article_markdown, output_file, word_count, meta_description,")
    print("  batch_id, created_at, updated_at, error_message")

## agents/ — Các AI Worker

Thư mục `agents/` chứa mỗi agent trong một file riêng, cùng với schema chung:

| File | Agent | Model | Tools | Output |
|------|-------|-------|-------|--------|
| `researcher.py` | Research Agent | Claude Sonnet | DuckDuckGo search | Văn bản thuần (ghi chú nghiên cứu) |
| `outliner.py` | Outline Agent | Claude Sonnet | Không | ContentOutline (schema có cấu trúc) |
| `writer.py` | Writer Agent | Grok-4 | Không | Bài viết Markdown thuần |
| `image.py` | Image Agent | Claude Sonnet | Freepik/DataForSEO | EnrichedContent (bài viết + hình ảnh) |
| `content_creator.py` | Content Creator | Claude Sonnet | workspace tools | Thành viên team chat |
| `status_tracker.py` | Status Tracker | Claude Sonnet | workspace tools | Thành viên team chat |
| `aio_analyst.py` | AIO Analyst | Claude Sonnet | workspace tools | Thành viên team chat |
| `team.py` | Tập hợp Team | Claude Opus (leader) | — | Agno Team |
| `schemas.py` | — | — | — | Pydantic models (ContentOutline, EnrichedContent, v.v.) |

**Tại sao dùng model khác nhau?**
- Claude Sonnet hỗ trợ `tools + output_schema` cùng lúc — hoàn hảo cho nghiên cứu và dàn ý
- Grok-4 được dùng cho viết bài vì tạo nội dung dài chất lượng cao
- Image Agent là **tuỳ chọn** — nếu không có API key cho hình ảnh, nó trả về `None` và pipeline bỏ qua bước làm giàu

**Một agent = một file.** Muốn thay đổi writer? Mở `writer.py`. Muốn thêm agent kiểm tra sự thật? Tạo `fact_checker.py`.

## tools/ — Các API bên ngoài

Thư mục `tools/` chứa các module kết nối với dịch vụ bên ngoài và workspace tools:

| File | Mục đích |
|------|----------|
| `airtable.py` | Database chính — tạo/đọc/cập nhật bài viết và phân tích AIO |
| `workspace.py` | Tools cho thành viên team chat (cầu nối tới pipeline + Airtable) |
| `aio.py` | Phân tích AI Overview qua DataForSEO |

Tất cả đều được thiết kế để **fallback an toàn**:
- Không có Airtable key → hàm báo lỗi rõ ràng
- Không có DataForSEO → phân tích AIO trả về kết quả rỗng

Lưu ý: Image toolkit (`FreepikTools`, `DataForSEOTools`) được định nghĩa trong `agents/image.py` cùng với image agent sử dụng chúng, không phải trong thư mục `tools/`. Như vậy dependency của mỗi agent nằm ngay cạnh agent đó.

## Tại Sao Cấu Trúc File Như Vậy?

Bạn có thể tự hỏi: sao không bỏ hết vào một file? Hoặc sao không tổ chức theo tính năng? Đây là triết lý đằng sau codebase này:

**Một file = một việc.** `researcher.py` nghiên cứu. `writer.py` viết bài. `airtable.py` nói chuyện với Airtable. Bạn không bao giờ phải đoán thứ gì nằm ở đâu.

**Một agent = một file.** Muốn thay đổi instructions của writer? Mở `agents/writer.py`. Muốn thêm agent kiểm tra sự thật? Tạo `agents/fact_checker.py`, import nó trong `__init__.py`, và nối vào pipeline. Bạn không cần hiểu toàn bộ codebase để thay đổi.

**Đặt tên = điều hướng.** Tên file cho bạn biết bên trong có gì mà không cần mở ra. Ai đó nói "dàn ý sai", bạn biết mở `agents/outliner.py`. Ai đó nói "bài viết không lưu được", bạn nhìn vào `tools/airtable.py`.

**`agents/` vs `tools/`** — Định nghĩa agent AI vs tích hợp dịch vụ bên ngoài. Agent là tầng "suy nghĩ" (dùng LLM). Tools là tầng "thực thi" (gọi API, đọc file, truy vấn database).

**`__init__.py` như một thư mục** — File `agents/__init__.py` re-export mọi thứ, nên `from agents import research_agent` hoạt động dù agents là một file hay cả một thư mục. Điều này có nghĩa là code *sử dụng* agents (như `pipeline.py`) không cần thay đổi khi bạn tái tổ chức.

## Bản đồ file hoàn chỉnh

Đây là mọi file trong `output/` và chức năng của nó:

```
output/
├── chat.py                  ← Điểm vào (~60 dòng, xác thực + khởi chạy)
├── pipeline.py              ← Điều phối (nghiên cứu → dàn ý → viết → làm giàu)
├── agents/
│   ├── __init__.py          ← Re-export mọi thứ (from agents import X vẫn hoạt động)
│   ├── schemas.py           ← Pydantic models (ContentOutline, EnrichedContent, v.v.)
│   ├── researcher.py        ← Research Agent (Claude Sonnet + DuckDuckGo)
│   ├── outliner.py          ← Outline Agent (Claude Sonnet + structured output)
│   ├── writer.py            ← Writer Agent (Grok-4, Markdown thuần)
│   ├── image.py             ← Image Agent + FreepikTools + DataForSEOTools
│   ├── content_creator.py   ← Thành viên team chat (tạo bài viết)
│   ├── status_tracker.py    ← Thành viên team chat (truy vấn bài viết)
│   ├── aio_analyst.py       ← Thành viên team chat (phân tích AIO)
│   └── team.py              ← Tập hợp Agno Team (Opus leader + 3 thành viên)
└── tools/
    ├── __init__.py          ← Package marker
    ├── airtable.py          ← Database chính (Airtable CRUD)
    ├── workspace.py         ← Hàm cầu nối cho team chat tools
    └── aio.py               ← Phân tích AI Overview qua DataForSEO
```

**16 file Python**, mỗi file có đúng một nhiệm vụ. Khi có lỗi, tên file cho bạn biết cần nhìn vào đâu.

## Bài Tập

Mở `output/pipeline.py` trong trình soạn văn bản (hoặc đọc nó trong ô code phía trên) và trả lời các câu hỏi sau:

1. Tìm 4 lệnh gọi agent (`research_agent.run(...)`, `outline_agent.run(...)`, v.v.). Mỗi agent nhận đầu vào gì?
2. Hàm nào được gọi giữa mỗi bước để cập nhật database?
3. Chuyện gì xảy ra nếu image agent là `None`?
4. File bài viết cuối cùng được lưu ở đâu? Tên file được xác định thế nào?

Đây là câu hỏi **chỉ đọc** — không tốn API. Bạn đang đọc code, giống như cách Claude Code khám phá codebase.

<details>
<summary>Bấm để xem đáp án</summary>

1. **Đầu vào của agent:**
   - `research_agent` nhận: chuỗi chủ đề
   - `outline_agent` nhận: ghi chú nghiên cứu từ bước 1
   - `writer_agent` nhận: outline JSON từ bước 2
   - `image_agent` nhận: bài viết markdown từ bước 3

2. **Giữa mỗi bước:** `update_article_status(article_id, status, **fields)` được gọi để lưu tiến trình vào Airtable.

3. **Nếu image agent là None:** Pipeline bỏ qua hoàn toàn bước làm giàu (`if image_agent is not None:`) và sử dụng output của writer làm bài viết cuối cùng.

4. **Vị trí file:** Lưu vào thư mục `content/`. Định dạng tên file: `{slug}-{article_id}-{timestamp}.md` trong đó slug là chủ đề viết thường với dấu gạch ngang.
</details>

## Tóm tắt

Mọi thứ kết nối qua **lời gọi hàm Python đơn giản**. Không có phép thuật.

- **`chat.py`** — Cửa vào. Xác thực key, import team, khởi chạy chat.
- **`agents/team.py`** — Team Leader (Opus) ủy thác cho 3 thành viên Sonnet.
- **`agents/*.py`** — Một file mỗi agent. Pipeline agent + thành viên team chat + schema.
- **`tools/workspace.py`** — Cầu nối. Đóng gói lệnh gọi pipeline/Airtable thành hàm tool đơn giản.
- **`pipeline.py`** — Bộ máy. Chạy 4 agent tuần tự với cập nhật Airtable ở mỗi bước.
- **`tools/airtable.py`** — Kho lưu trữ. Database Airtable với các hàm trả về dict.
- **`tools/aio.py`** — Phân tích AI Overview qua DataForSEO.

Toàn bộ luồng: **chat.py → agents/team.py → tools/workspace.py → pipeline.py → agents/*.py → tools/airtable.py**

Triết lý: **một file = một việc, một agent = một file, đặt tên = điều hướng.**

**Bài tiếp theo:** Chúng ta sẽ xem giao diện chat hoạt động thực tế và tìm hiểu cách workspace tools kết nối team với mọi thứ bạn vừa thấy ở đây!